# NB_SLV_POI_ENTITY

RECOMPUTE `lh_vv_silver.dbo.slv_poi_entity`. Chỉ dòng google. `created_at` / `updated_at` là vòng đời dòng entity.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
GOOGLE = "google"
OTHER = "other"
CATEGORY_COLS = ("business_sector", "business_category", "business_category_vi")


def build(ctx: NodeContext) -> DataFrame:
    # Giữ mọi cột slv_3p_poi trừ created_at / updated_at (lib thêm cột kỹ thuật của bảng đích).
    # Khớp: primary_type -> phần tử đầu tiên của types_json khớp (theo vị trí mảng) -> other / other / NULL.
    src = ctx.input("poi")
    clash = [c for c in src.columns if c.lower() in CATEGORY_COLS + ("row_hash", "deleted_at")]
    if clash:
        raise ConfigError(f"slv_poi_entity: slv_3p_poi đã có cột {clash}")
    keep = [c for c in src.columns if c.lower() not in TECH_NAMES]
    poi = src.where(F.lower(F.trim("source_name")) == GOOGLE).select(*keep)
    d = (ctx.input("dict")
         .where((F.lower(F.trim("source_name")) == GOOGLE) & (F.col("is_active") == F.lit(True)))
         .select(F.lower(F.trim("place_type")).alias("_k"), F.struct(*CATEGORY_COLS).alias("_cat")))
    dup = d.groupBy("_k").count().where("count > 1").count()
    if dup:
        raise QualityGateError(f"ref_dictionary_place_type: {dup} place_type google trùng")
    d = F.broadcast(d)
    by_primary = (poi.select("poi_id", F.lower(F.trim("primary_type")).alias("_k"))
                  .join(d, "_k").select("poi_id", F.col("_cat").alias("_p")))
    by_types = (poi.select("poi_id", F.posexplode(F.from_json("types_json", "array<string>")).alias("_pos", "_t"))
                .withColumn("_k", F.lower(F.trim("_t"))).join(d, "_k")
                .groupBy("poi_id").agg(F.min_by("_cat", "_pos").alias("_m")))
    out = poi.join(by_primary, "poi_id", "left").join(by_types, "poi_id", "left")
    agg = out.agg(F.count(F.lit(1)).alias("rows"), F.count("_p").alias("by_primary"),
                  F.count(F.when(F.col("_p").isNull(), F.col("_m"))).alias("by_types")).first()
    total = src.count()
    ctx.metrics.update(rows=agg["rows"], by_primary=agg["by_primary"], by_types=agg["by_types"],
                       unmatched=agg["rows"] - agg["by_primary"] - agg["by_types"],
                       non_google_rows=total - agg["rows"], dict_google_rows=d.count())
    if total and not agg["rows"]:
        raise QualityGateError("slv_poi_entity: slv_3p_poi không có dòng google")

    def pick(c, default):
        return (F.when(F.col("_p").isNotNull(), F.col(f"_p.{c}"))
                .when(F.col("_m").isNotNull(), F.col(f"_m.{c}"))
                .otherwise(default).alias(c))

    return out.select(*keep, pick("business_sector", F.lit(OTHER)), pick("business_category", F.lit(OTHER)),
                      pick("business_category_vi", F.lit(None).cast("string")))


SPEC = NodeSpec(
    trg_tbl="slv_poi_entity",
    inputs={"poi": "slv_3p_poi", "dict": "ref_dictionary_place_type"},
    pk=("poi_id",),
    build=build,
    required_inputs=("poi", "dict"),
    warn_checks=(("types_json_unparsed",
                  "types_json IS NOT NULL AND from_json(types_json, 'array<string>') IS NULL"),),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
